In [ ]:
import pandas as pd
import numpy as np


from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from rank_bm25 import BM25Okapi

file_path = '/content/drive/MyDrive/DataSet/indonesia_news_1000.csv'
df = pd.read_csv(file_path)
df.head(10)

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source,Dokumen_Asli,Cleaned_Content
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",viral isu phk buruh gudang garam said iqbal su...
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara Gem...",gempa m guncang pulau doi malu utara gempa bum...
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...",toko emas palsu riau bongkar polisi hias kg si...
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,minyakita tak sesuai ukur temu kudus sita pold...
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",pimpin ldp sanae takaichi calon kuat pm peremp...
5,Gubernur Banten Terpilih Andra Soni Temui Pres...,"Jumat, 13 Des 2024 22:12 WIB",https://news.detik.com/berita/d-7685816/gubern...,Presiden Prabowo Subianto bertemu dengan Guber...,prabowo subianto,andra soni,istana negara,NaN,NaN,detik,Gubernur Banten Terpilih Andra Soni Temui Pres...,gubernur banten pilih andra soni temu presiden...
6,Yusril: Pedemo Ditangkap Polisi karena Lakukan...,27 September 2025 | 08.29 WIB,https://www.tempo.co/hukum/yusril-pedemo-ditan...,"Baca berita dengan sedikit iklan, klik di sin...",yusril,demonstrasi,demonstran,penangkapan-aktivis,yusril-ihza-mahendra,tempo,Yusril: Pedemo Ditangkap Polisi karena Lakukan...,yusril demo tangkap polisi laku tindak pidana ...
7,"Banjir Besar Landa Jabodetabek, Pimpinan MPR I...",4 Maret 2025,https://nasional.kompas.com/read/2025/03/04/22...,"JAKARTA, KOMPAS.com - Wakil Ketua MPR RI Eddy...",eddy soeparno,mitigasi bencana,banjir jabodetabek,krisis iklim,Banjir Jabodetabek hari ini,kompas,"Banjir Besar Landa Jabodetabek, Pimpinan MPR I...",banjir besar landa jabodetabek pimpin mpr inga...
8,"Lagu ""Indonesia Raya"" Kena Royalti? Istana Men...",15 Agustus 2025,https://nasional.kompas.com/read/2025/08/15/18...,"JAKARTA, KOMPAS.com - Menteri Sekretaris Nega...",royalti,istana,Indonesia Raya,PSSI,Prasetyo Hadi,kompas,"Lagu ""Indonesia Raya"" Kena Royalti? Istana Men...",lagu indonesia raya kena royalti istana jawab ...
9,Remaja Ditindak Polisi di Serpong karena Bawa ...,29 April 2025,https://megapolitan.kompas.com/read/2025/04/29...,"TANGERANG SELATAN, KOMPAS.com - Seorang remaj...",polisi,remaja,Mobil berpelat asing,Remaja ditindak polisi,Gunakan mobil pelat asing,kompas,Remaja Ditindak Polisi di Serpong karena Bawa ...,remaja tindak polisi serpong bawa mobil pelat ...


In [ ]:
pip install rank_bm25

In [ ]:
# 1. PERSIAPAN DATA
# Kita pastikan tidak ada data kosong di kolom hasil stemming
df = df.dropna(subset=['Cleaned_Content'])
print(f"✅ Data siap diproses: {len(df)} dokumen.")

# 2. MEMBUAT MODEL TF-IDF (Vector Space Model)
# Ini mengubah kata-kata di 'Cleaned_Content' menjadi matriks angka
print("Sedang membuat matriks TF-IDF...")
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Content'])
print(f"✅ Model selesai! Ukuran Matrix: {tfidf_matrix.shape}")

# 3. SIAPKAN STEMMER UNTUK QUERY
# Kita butuh ini agar kalau user cari "Kenaikan", mesin mengubahnya jadi "naik" (cocok dengan data)
factory = StemmerFactory()
stemmer_query = factory.create_stemmer()

# 4. FUNGSI PENCARIAN (SEARCH ENGINE)
def search(query, top_k=10):
    print(f"\nMencari: '{query}'")

    # a. Preprocessing Query (Samakan perlakuan dengan data)
    query_clean = query.lower() # Huruf kecil
    query_stemmed = stemmer_query.stem(query_clean) # Stemming query
    print(f"(Kata dasar pencarian: '{query_stemmed}')")

    # b. Ubah query jadi vektor angka
    query_vec = vectorizer.transform([query_stemmed])

    # c. Hitung kemiripan (Cosine Similarity)
    # Bandingkan vektor query dengan seluruh dokumen
    cosine_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # d. Ambil dokumen dengan skor tertinggi
    top_indices = cosine_scores.argsort()[-top_k:][::-1]

    # e. Tampilkan Hasil
    print("-" * 60)
    found = False
    for rank, index in enumerate(top_indices, 1):
        score = cosine_scores[index]
        if score > 0.001: # Hanya tampilkan jika ada kemiripan
            found = True
            judul = df.iloc[index]['Judul']
            # Potong isi berita biar rapi
            isi = str(df.iloc[index]['Content'])[:200].replace('\n', ' ')

            print(f"Rank {rank} | Score: {score:.4f}")
            print(f"Judul: {judul}")
            print(f"Isi  : {isi}...")
            print("-" * 60)

    if not found:
        print("❌ Maaf, tidak ditemukan dokumen yang relevan.")

# Langsung tes cari berita di sini
search("harga")

✅ Data siap diproses: 1000 dokumen.
Sedang membuat matriks TF-IDF...
✅ Model selesai! Ukuran Matrix: (1000, 15614)

Mencari: 'harga'
(Kata dasar pencarian: 'harga')
------------------------------------------------------------
Rank 1 | Score: 0.3616
Judul: Prediksi Harga Emas Sepekan ke Depan
Isi  : Baca berita dengan sedikit iklan,  klik di sini  HARGA emas dunia ditutup menguat di level US$ 3.684,38 per troy ounce pada perdagangan Jumat, 19 September 2025. Direktur Traze Andalan Futures Ibrahim...
------------------------------------------------------------
Rank 2 | Score: 0.2989
Judul: KAI Bantah Harga Tiket Kereta Api Naik Usai Lebaran 2025
Isi  :  JAKARTA, KOMPAS.com - PT Kereta Api Indonesia (Persero) atau KAI Daop 1 Jakarta membantah harga tiket kereta api naik setelah Lebaran 2025. Manajer Humas KAI Daop 1 Jakarta, Ixfan Hendriwintoko, meng...
------------------------------------------------------------
Rank 3 | Score: 0.2577
Judul: Menteri Amran Bantah Beras Mahal Akibat Bulog 

In [ ]:
# 1. DAFTAR 10 QUERY (Mencakup berbagai topik di datasetmu)
# Kamu boleh ganti kata-katanya kalau mau, tapi ini sudah saya sesuaikan dengan topik umum berita
queries = [
    "kenaikan harga bahan pokok",       # Ekonomi
    "gempa bumi dan tsunami",           # Bencana
    "timnas indonesia sepak bola",      # Olahraga
    "kasus korupsi pejabat",            # Hukum/Politik
    "investasi saham dan emas",         # Ekonomi/Bisnis
    "banjir di jakarta",                # Bencana
    "pemutusan hubungan kerja phk",     # Ketenagakerjaan
    "wisata kuliner indonesia",         # Lifestyle/Travel
    "kecelakaan lalu lintas tol",       # Peristiwa
    "pilkada serentak 2024"             # Politik
]

# 2. WADAH PENAMPUNG HASIL
data_untuk_labeling = []

print("Sedang menyiapkan data untuk Ground Truth...")
print("-" * 50)

# 3. LOOPING: Jalankan setiap query menggunakan BM25
# (Kita pakai BM25 untuk generate kandidat karena biasanya lebih akurat dari TF-IDF)
for id_q, q in enumerate(queries, 1):
    print(f"Processing Query {id_q}: {q}")

    # a. Preprocessing Query
    q_clean = q.lower()
    try:
        q_stem = stemmer_query.stem(q_clean)
    except:
        q_stem = q_clean

    # b. Search pakai BM25 (Ambil Top 10)
    tokenized_query = q_stem.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[-10:][::-1] # Ambil 10 teratas

    # c. Simpan ke list
    for rank, idx in enumerate(top_indices, 1):
        judul = df.iloc[idx]['Judul']
        # Ambil cuplikan isi berita (200 huruf) buat bantuin kamu baca cepat
        isi_singkat = str(df.iloc[idx]['Content'])[:250].replace('\n', ' ')

        data_untuk_labeling.append({
            'Query_ID': id_q,
            'Query_Text': q,
            'Rank_BM25': rank, # Peringkat menurut mesin
            'Judul_Berita': judul,
            'Isi_Singkat': isi_singkat,
            'Relevan': 0 # <--- INI NANTI KAMU ISI DI EXCEL (0 atau 1)
        })

# 4. SIMPAN KE CSV UNTUK DIDOWNLOAD
df_labeling = pd.DataFrame(data_untuk_labeling)
nama_file_excel = 'ground_truth_SIAP_ISI.csv'
df_labeling.to_csv(nama_file_excel, index=False)

print("-" * 50)
print(f"✅ SELESAI! File '{nama_file_excel}' berhasil dibuat.")
print("Silakan download file tersebut di panel sebelah kiri (icon folder).")

Sedang menyiapkan data untuk Ground Truth...
--------------------------------------------------
Processing Query 1: kenaikan harga bahan pokok
Processing Query 2: gempa bumi dan tsunami
Processing Query 3: timnas indonesia sepak bola
Processing Query 4: kasus korupsi pejabat
Processing Query 5: investasi saham dan emas
Processing Query 6: banjir di jakarta
Processing Query 7: pemutusan hubungan kerja phk
Processing Query 8: wisata kuliner indonesia
Processing Query 9: kecelakaan lalu lintas tol
Processing Query 10: pilkada serentak 2024
--------------------------------------------------
✅ SELESAI! File 'ground_truth_SIAP_ISI.csv' berhasil dibuat.
Silakan download file tersebut di panel sebelah kiri (icon folder).
